# Per-subject dimensionality reduction of gait data

Fits a per-subject decomposition (PCA or NMF) to one of three movement
representations and exports the resulting component scores, averaged within each
task condition, as a tidy CSV.

**Modalities (`MODE`)**

| `MODE` | Input | Features |
| --- | --- | --- |
| `kinematics` | stride-normalized joint angles (parquet, external) | hip / knee / ankle / lumbar angles |
| `emg` | gait-cycle-normalized surface EMG (parquet, external) | TA, SOL, MG, VM, RF, BF |
| `gait_metrics` | spatiotemporal summaries (`data/data_BMH*.xlsx`) | speed, step width / length and their variability, foot-placement error, head angle |

**Conditions.** Each trial is labelled `sXaYbZ`, where `X`, `Y`, `Z` are the
speed, accuracy and balance prompt levels (0 = low, 1 = medium, 2 = high).

**Output.** `data/decomp_<mode>_<method>_<config>.csv`, one row per
subject x condition with columns `Comp1 ... CompK`. These files are the inputs to
`01_estimate_objective_weights.ipynb`.

**If you do run it.** Set the options in the *Configuration* cell, then run the
notebook top to bottom, once per modality. Only `MODE = "gait_metrics"` works
from the data in this repository; see the note below.

> ## Reference notebook — not part of the run order
>
> This is the record of **how** the committed feature files were produced: the
> number of components, the per-subject fitting, the left/right averaging, the
> excluded conditions. `01_estimate_objective_weights.ipynb` and
> `02_prompt_effects_lmm.ipynb` read those committed files and do not need this
> notebook or the raw data.
>
> It is executable if you want to check the pipeline. `MODE = "gait_metrics"`
> (the default) runs from the data in this repository and regenerates
> `data/decomp_gait_metrics_pca_fixed5.csv`. `MODE = "kinematics"` and
> `MODE = "emg"` read stride-level parquet files that are not distributed here
> (see *Data availability* in the README): without them the load cell reports
> what is missing and the remaining cells no-op instead of failing, so nothing
> is overwritten.


## Setup

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import resample
from sklearn.decomposition import PCA, NMF

# ---------------- Paths ---------------- #
# Paths are relative to the repository root, so launch Jupyter from there.

DATA_DIR = Path("data")

# Decomposition outputs are written here, alongside the input data.
OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Per-trial survey responses; also used to map each (speed, accuracy, balance)
# prompt combination to the trial ID that was collected under it.
SUBJECTIVE_XLS = DATA_DIR / "Subjective_Responses.xlsx"

# Spatiotemporal gait metrics, one workbook per subject (data/data_BMH*.xlsx).
GAIT_METRICS_BASE = DATA_DIR

# Stride-level kinematics and EMG are too large to version, so they live outside
# the repository. Point MO_RAW_DATA_DIR at a folder holding one subfolder per
# subject ("BMH01/", "BMH02/", ...), each containing:
#   <subject>_cleaned_strides.parquet        paired joint angles
#   <subject>_cleaned_strides_extra.parquet  unpaired joint angles (lumbar, ...)
#   *emg*.parquet                            normalized, cycle-segmented EMG
# MODE = "gait_metrics" runs without it.
RAW_DATA_DIR = Path(os.environ.get("MO_RAW_DATA_DIR", "raw_data"))

# ---------------- Subjects ---------------- #

KIN_SUBJECTS = [
    "BMH01", "BMH02", "BMH06", "BMH07", "BMH08", "BMH09",
    "BMH10", "BMH13", "BMH17", "BMH19", "BMH20", "BMH21",
]

# Minimum samples (strides or time points) a subject needs before PCA/NMF is run.
MIN_SAMPLES_PER_SUBJECT = 10

# ---------------- Kinematics config ---------------- #

# Left/right pairs are averaged into a single side-agnostic channel.
JOINT_PAIRS = [
    ("hip_flexion_r",   "hip_flexion_l"),
    ("hip_adduction_r", "hip_adduction_l"),
    ("knee_angle_r",    "knee_angle_l"),
    ("ankle_angle_r",   "ankle_angle_l"),
]

# Midline joints, used as-is.
unpaired_cols = ["lumbar_bending", "lumbar_extension"]

# ---------------- EMG config ---------------- #

BASE_MUSCLES = ["TA", "SOL", "MG", "VM", "RF", "BF"]

# ---------------- Gait-metrics config --------------- #

GAIT_METRICS_FILES = [f"data_{sub}.xlsx" for sub in KIN_SUBJECTS]

# Prompt levels are stored as words in the workbooks; map them to 0/1/2.
WALKING_SPEED_MAPPING = {"Slow": 0, "Medium": 1, "Fast": 2}
ACCURACY_MAPPING      = {"Low": 0, "Medium": 1, "High": 2}
BALANCE_MAPPING       = {"Low": 0, "Medium": 1, "High": 2}

# "Condition" is kept in the list so the loader can split it off from the features.
GAIT_PREDICTOR_COLUMNS = [
    "Mean Error Straights",
    "Mean Width Straights (mm)",
    "Straights Width Variability (mm)",
    "Mean Length Straights (mm)",
    "Straights Length Variability (mm)",
    "Average Speed (m/s)",
    "Head Angle (deg)",
    "Condition",
]

## Configuration

In [ ]:
# CONFIG
# =============================================================================

# Choose data mode: "kinematics", "gait_metrics", or "emg".
# Only "gait_metrics" runs from the data in this repository; the other two need
# the stride-level parquet files pointed to by MO_RAW_DATA_DIR.
MODE = "gait_metrics"

# Decomposition method: "pca" or "nmf"
DECOMP_METHOD = "pca"

# either use a variance threshold OR a fixed number of components
USE_FIXED_COMPONENTS = True         # if True, ignore VARIANCE_THRESHOLD and use N_FIXED_COMPONENTS
N_FIXED_COMPONENTS = 5              # only used if USE_FIXED_COMPONENTS = True
VARIANCE_THRESHOLD = 0.8            # if USE_FIXED_COMPONENTS is False, choose number of exponents for VAF threshold, ensuring at least MIN_COMPONENTS
MIN_COMPONENTS = 2                  # keep at least this number of components even if VAF threshold is met to ensure not overdetermined

# How to represent EMG for NMF/PCA
#   "synergy" -> channels as features, time×strides as samples (classic EMG synergies)
#   "stride"  -> strides as samples, time×channels as features (current stridewise setup)
COLLAPSE_MODE = "synergy"

## Decomposition helpers

`run_decomposition()` is the single entry point; it dispatches to PCA or NMF and
applies the component-count policy chosen above (fixed *K*, or the smallest *K*
reaching `VARIANCE_THRESHOLD`).

In [ ]:
# =============================================================================
# DECOMPOSITION HELPERS
# =============================================================================

def run_pca_with_policy(
    X,
    use_fixed_components,
    n_fixed_components,
    variance_threshold,
    min_components,
):
    """
    PCA respecting unified config:

    - If use_fixed_components = True:
        keep exactly n_fixed_components (clipped to <= min(n_samples, n_features))

    - Else:
        choose the smallest number of PCs whose cumulative explained variance
        >= variance_threshold, but keep at least min_components.
    """
    n_samples, n_features = X.shape
    max_components = min(n_samples, n_features)

    if use_fixed_components:
        n_keep = int(n_fixed_components)
        if n_keep < 1:
            raise ValueError("N_FIXED_COMPONENTS must be >= 1")
        n_keep = min(n_keep, max_components)

        pca = PCA(n_components=n_keep)
        X_scores = pca.fit_transform(X)
        var_explained = pca.explained_variance_ratio_

        return pca, X_scores, n_keep, var_explained

    # --- variance-threshold mode ---
    if variance_threshold is None:
        raise ValueError("variance_threshold must be provided when not using fixed components.")

    # Fit full PCA to get entire spectrum
    pca_full = PCA(n_components=max_components)
    X_scores_all = pca_full.fit_transform(X)
    var_explained = pca_full.explained_variance_ratio_
    cum_var = np.cumsum(var_explained)

    n_keep = int(np.searchsorted(cum_var, variance_threshold) + 1)
    # enforce minimum
    n_keep = max(n_keep, int(min_components))
    n_keep = min(n_keep, max_components)

    X_scores = X_scores_all[:, :n_keep]
    return pca_full, X_scores, n_keep, var_explained


def run_nmf_with_policy(
    X,
    use_fixed_components,
    n_fixed_components,
    min_components,
    variance_threshold,
):
    """
    NMF respecting unified config.

    - If use_fixed_components = True:
        => n_components = n_fixed_components (clipped to <= min(n_samples, n_features))

    - Else (search mode):
        => search over K from min_components..K_max and choose the smallest K
           such that VAF(K) >= variance_threshold.
           If no K reaches the threshold, use K_max.

    VAF(K) is defined as: 1 - ||X - W_K H_K||_F^2 / ||X||_F^2

    Returns
    -------
    nmf : fitted NMF model
    W   : scores, shape (n_samples, K)
    K   : chosen number of components
    vaf_global : float, global VAF for chosen K
    """

    n_samples, n_features = X.shape

    # Set any negative values to zero (NMF requires non-negativity)
    X_nonneg = X.copy()
    X_nonneg[X_nonneg < 0] = 0

    k_max = min(n_samples, n_features)
    if k_max < 1:
        raise ValueError("Not enough samples/features to fit NMF (k_max < 1).")
    
    # Precompute norm of X for VAF
    norm_X2 = np.linalg.norm(X_nonneg, ord="fro") ** 2

    # ---------- fixed-K mode ----------
    if use_fixed_components:
        k = int(n_fixed_components)
        if k < 1:
            raise ValueError("N_FIXED_COMPONENTS must be >= 1")
        k = min(k, k_max)

        nmf = NMF(
            n_components=k,
            init="nndsvda",
            random_state=0,
            max_iter=1000,
        )
        W = nmf.fit_transform(X_nonneg)
        H = nmf.components_

        recon = W @ H
        err2 = np.linalg.norm(X_nonneg - recon, ord="fro") ** 2
        vaf_global = 1.0 - err2 / norm_X2

        return nmf, W, k, vaf_global

    # ---------- search-over-K mode ----------
    if variance_threshold is None:
        raise ValueError("variance_threshold must be provided when not using fixed components.")

    k_start = max(1, int(min_components))
    k_start = min(k_start, k_max)

    best_k = None
    best_model = None
    best_W = None
    best_vaf = -np.inf

    for k in range(k_start, k_max + 1):
        nmf = NMF(
            n_components=k,
            init="nndsvda",
            random_state=0,
            max_iter=1000,
        )
        W = nmf.fit_transform(X_nonneg)
        H = nmf.components_
        recon = W @ H

        err2 = np.linalg.norm(X_nonneg - recon, ord="fro") ** 2
        vaf_k = 1.0 - err2 / norm_X2

        # keep track of the best K seen so far
        if vaf_k > best_vaf:
            best_vaf = vaf_k
            best_k = k
            best_model = nmf
            best_W = W

        if vaf_k >= variance_threshold:
            # smallest K that reaches the VAF threshold
            break

    # best_model / best_W / best_k / best_vaf correspond to either:
    # - first K that met the threshold, or
    # - if none met it, the K with highest VAF up to k_max.
    return best_model, best_W, best_k, best_vaf


def run_decomposition(
    X,
    method,
    use_fixed_components=USE_FIXED_COMPONENTS,
    n_fixed_components=N_FIXED_COMPONENTS,
    variance_threshold=VARIANCE_THRESHOLD,
    min_components=MIN_COMPONENTS,
):
    """
    Unified interface:

    Parameters
    ----------
    X : array, shape (n_samples, n_features)
    method : "pca" or "nmf"

    Uses global config:
      USE_FIXED_COMPONENTS, N_FIXED_COMPONENTS, VARIANCE_THRESHOLD, MIN_COMPONENTS

    Returns
    -------
    model : PCA or NMF object
    X_scores : array, shape (n_samples, n_kept)
    n_kept : int
    var_explained : array or None
    """
    method = method.lower()
    if method == "pca":
        return run_pca_with_policy(
            X,
            use_fixed_components=use_fixed_components,
            n_fixed_components=n_fixed_components,
            variance_threshold=variance_threshold,
            min_components=min_components,
        )
    elif method == "nmf":
        return run_nmf_with_policy(
            X,
            use_fixed_components=use_fixed_components,
            n_fixed_components=n_fixed_components,
            variance_threshold=variance_threshold,
            min_components=min_components,
        )
    else:
        raise ValueError(f"Unknown method: {method}")

## Condition bookkeeping

`build_trial_names_by_condition()` reads the survey workbook and returns, for
every (speed, accuracy, balance) combination, the trial each subject performed
under it. `aggregate_condition_means()` collapses sample-level component scores
into one row per subject x condition.

In [ ]:
# =============================================================================
# CONDITION HELPERS
# =============================================================================

def aggregate_condition_means(subject_id, X_scores, cond_labels):
    """
    Given component scores for all samples for one subject and their condition labels,
    compute mean scores per condition.

    Assumes condition labels like "s2a0b1".
    """
    cond_labels = np.array(cond_labels)
    unique_conds = np.unique(cond_labels)
    n_pcs = X_scores.shape[1]

    rows = []
    for cond in unique_conds:
        mask = (cond_labels == cond)
        X_cond = X_scores[mask, :]
        n_samples_cond = X_cond.shape[0]
        mean_scores = X_cond.mean(axis=0)

        # Parse cond like "s2a0b1"
        speed_val = np.nan
        acc_val = np.nan
        bal_val = np.nan
        try:
            speed_val = int(cond.split("a")[0][1:])
            acc_val = int(cond.split("a")[1].split("b")[0])
            bal_val = int(cond.split("b")[1])
        except Exception:
            pass

        row = {
            "Subject": subject_id,
            "Condition": cond,
            "Speed": speed_val,
            "Target": acc_val,
            "Balance": bal_val,
            "n_samples": n_samples_cond,
        }
        for i in range(n_pcs):
            row[f"Comp{i+1}"] = mean_scores[i]   # generic "Comp" works for PCA or NMF

        rows.append(row)

    return rows


def build_trial_names_by_condition(subjective_xls, subjects):
    """
    From Subjective_Responses.xlsx, build
        target_conditions: list of (speed, target, balance)
        trial_names_by_condition[(s, a, b)][subject] = "Trial000X"
    """
    xls = pd.ExcelFile(subjective_xls)

    dfs = {}
    for sub in subjects:
        df = pd.read_excel(xls, sheet_name=sub)
        df.columns = df.columns.str.strip()
        df = df.iloc[2:29].copy()
        df["Subject"] = sub
        for col in ["Speed Prompt", "Target Prompt", "Balance Prompt"]:
            df[col] = df[col].astype(float).astype(int)
        dfs[sub] = df[["Trial", "Speed Prompt", "Target Prompt", "Balance Prompt", "Subject"]]

    df_all = pd.concat(dfs.values(), ignore_index=True)

    target_conditions = list(itertools.product([0, 1, 2], repeat=3))
    trial_names_by_condition = {}
    for cond in target_conditions:
        speed_val, acc_val, bal_val = [int(x) for x in cond]
        trial_names = {}
        for sub in subjects:
            df_sub = df_all[df_all["Subject"] == sub]
            match = df_sub[
                (df_sub["Speed Prompt"] == speed_val)
                & (df_sub["Target Prompt"] == acc_val)
                & (df_sub["Balance Prompt"] == bal_val)
            ]
            if match.empty:
                continue
            trial_num = int(match["Trial"].iloc[0])
            trial_name = f"Trial{trial_num:04d}"
            trial_names[sub] = trial_name
        trial_names_by_condition[cond] = trial_names

    return target_conditions, trial_names_by_condition

## Data loaders

One loader per modality. Each returns `data_by_subject[subject]` with

* `X` — the data matrix passed to PCA/NMF,
* `cond_labels` — one `sXaYbZ` label per row of `X`,
* `feature_names` — column names, where they are well defined.

`COLLAPSE_MODE` decides how strides become rows: `"synergy"` stacks time points
(rows = time, columns = channels, the classic synergy layout), while `"stride"`
uses one row per stride with time x channel flattened into the columns.

In [ ]:
# =============================================================================
# KINEMATICS LOADERS
# =============================================================================

def load_all_kinematics_strides():
    """
    Load all subjects' cleaned strides from RAW_DATA_DIR into one DataFrame.
    For each subject, loads:
      - <name>_cleaned_strides.parquet        (paired joints: hip, knee, ankle, etc.)
      - <name>_cleaned_strides_extra.parquet  (unpaired joints: lumbar_bending, etc.)
    and merges them on shared key columns.
    """
    KEY_COLS = ["subject", "trial", "gc", "cycle_idx", "side"]

    bmh_folders = sorted(
        [d for d in RAW_DATA_DIR.iterdir() if d.is_dir() and d.name.startswith("BMH")]
    )
    print(f"Found {len(bmh_folders)} BMH folders")

    dfs = []
    for bmh_folder in bmh_folders:
        name = bmh_folder.name.lower()
        base_file  = bmh_folder / f"{name}_cleaned_strides.parquet"
        extra_file = bmh_folder / f"{name}_cleaned_strides_extra.parquet"

        if not base_file.exists() and not extra_file.exists():
            print(f"Warning: no parquet files found for {bmh_folder.name}. Skipping.")
            continue

        df_base  = pd.read_parquet(base_file)  if base_file.exists()  else None
        df_extra = pd.read_parquet(extra_file) if extra_file.exists() else None

        if df_base is None:
            print(f"Warning: base parquet missing for {bmh_folder.name}, using extra only.")
            dfs.append(df_extra)
        elif df_extra is None:
            print(f"Warning: extra parquet missing for {bmh_folder.name}, using base only.")
            dfs.append(df_base)
        else:
            merge_keys = [c for c in KEY_COLS if c in df_base.columns and c in df_extra.columns]
            extra_new_cols = [c for c in df_extra.columns if c not in df_base.columns or c in merge_keys]
            df_merged = df_base.merge(df_extra[extra_new_cols], on=merge_keys, how="left")
            dfs.append(df_merged)

    if not dfs:
        raise RuntimeError("No parquet files loaded.")

    return pd.concat(dfs, ignore_index=True)

def load_kinematics_subject_data(df_total, subjects, target_conditions,
                                 trial_names_by_condition, joint_pairs,
                                 unpaired_cols=None):
    """
    Returns:
        data_by_subject[subject] = {
            "X": ndarray (n_strides, n_features),
            "cond_labels": list of str, length n_strides (e.g. "s2a0b1")
            "stride_mats": list of (T, C) arrays, T=100,
                            C = len(joint_pairs) + len(unpaired_cols)
            "feature_names": list of str, length T*C
        }

        Kinematics are 100 time points per stride.
        For columns in `joint_pairs` we average L/R; for `unpaired_cols`
        we use the column as-is (no L/R pairing).
    """
    if unpaired_cols is None:
        unpaired_cols = []

    data_by_subject = {}

    required_cols = {"subject", "trial", "gc", "cycle_idx", "side"}
    missing = required_cols - set(df_total.columns)
    if missing:
        raise ValueError(f"df_total is missing required columns: {missing}")

    # Optional: check unpaired columns exist
    for col in unpaired_cols:
        if col not in df_total.columns:
            raise ValueError(f"unpaired column '{col}' not found in df_total")

    for sub in subjects:
        df_sub = df_total[df_total["subject"] == sub].copy()
        if df_sub.empty:
            print(f"{sub}: MISSING")
            continue

        stride_vectors = []
        stride_mats = []  # list of (T, C) matrices
        cond_labels = []
        n_nan_skipped = 0

        for cond in target_conditions:
            speed_val, acc_val, bal_val = cond
            cond_str = f"s{speed_val}a{int(acc_val)}b{int(bal_val)}"

            if (cond == (0, 2, 2) and sub in ("BMH01", "BMH02", "BMH08")) or \
               (cond == (1, 2, 2) and sub == "BMH08"):
                print(f"{sub}: SKIP CONDITION {cond}")
                continue

            trials_for_cond = trial_names_by_condition.get(cond, {})
            trial_name = trials_for_cond.get(sub, None)
            if trial_name is None:
                print(f"{sub}: TRIAL IS NONE {cond}")
                continue

            df_sel = df_sub[df_sub["trial"] == trial_name].copy()
            if df_sel.empty:
                print(f"{sub}: TRIAL EMPTY {cond}")
                continue

            df_R = df_sel[df_sel["side"] == "R"].sort_values("gc")
            df_L = df_sel[df_sel["side"] == "L"].sort_values("gc")

            for cyc_idx in df_R["cycle_idx"].unique():
                cyc_R = df_R[df_R["cycle_idx"] == cyc_idx].sort_values("gc")
                cyc_L = df_L[df_L["cycle_idx"] == cyc_idx].sort_values("gc")

                if cyc_R.shape[0] == 101 and cyc_L.shape[0] == 101:
                    # Original GC grid (0..100%, 101 points)
                    orig_gc = cyc_R["gc"].values.astype(float)

                    # New GC grid: 100 points from 0% to 100%
                    new_gc = np.linspace(0.0, 100.0, 100)

                    def _resolve_col(df, col_name):
                        """Return col_name if present; else strip _r/_l suffix and retry."""
                        if col_name in df.columns:
                            return col_name
                        for suffix in ("_r", "_l", "_R", "_L"):
                            if col_name.endswith(suffix):
                                base = col_name[:-len(suffix)]
                                if base in df.columns:
                                    return base
                        raise KeyError(
                            f"Column '{col_name}' (or base name) not found in DataFrame"
                        )

                    kin_data = []

                    # ---- L/R-PAIRED COLUMNS ----
                    for col_r, col_l in joint_pairs:
                        # Right side
                        y_r = cyc_R[_resolve_col(cyc_R, col_r)].values.astype(float)
                        y_r_resamp = np.interp(new_gc, orig_gc, y_r)

                        # Left side
                        y_l = cyc_L[_resolve_col(cyc_L, col_l)].values.astype(float)
                        y_l_resamp = np.interp(new_gc, orig_gc, y_l)

                        # Average across sides
                        y_avg = 0.5 * (y_r_resamp + y_l_resamp)
                        kin_data.append(y_avg)

                    # ---- UNPAIRED COLUMNS (e.g., lumbar_bending) ----
                    # Assumes these columns are defined once per sample.
                    # We take them from the R side; if you prefer, you can
                    # switch to df_sel[df_sel["cycle_idx"] == cyc_idx] instead.
                    for col in unpaired_cols:
                        if col in cyc_R.columns:
                            src = cyc_R
                        elif col in cyc_L.columns:
                            src = cyc_L
                        else:
                            raise KeyError(
                                f"Column '{col}' not found in R or L for "
                                f"{sub}, cond {cond}, cycle {cyc_idx}"
                            )

                        y = src[col].values.astype(float)
                        y_resamp = np.interp(new_gc, orig_gc, y)
                        kin_data.append(y_resamp)

                    # (100, C_total), C_total = len(joint_pairs) + len(unpaired_cols)
                    stride_data = np.column_stack(kin_data)

                    # Skip strides with missing data (e.g. from incomplete extra parquet merge)
                    if np.isnan(stride_data).any():
                        n_nan_skipped += 1
                        continue

                    stride_vec = stride_data.reshape(-1)

                    stride_vectors.append(stride_vec)
                    cond_labels.append(cond_str)
                    stride_mats.append(stride_data)  # now (100, C_total)

        if n_nan_skipped:
            print(f"{sub}: skipped {n_nan_skipped} strides with NaN")

        if not stride_vectors:
            print(f"{sub}: no valid strides for any condition, skipping.")
            continue

        X = np.vstack(stride_vectors)

        # ---------- build feature_names matching X ----------
        # Use the first stride matrix to infer time length and column ordering
        T, C = stride_mats[0].shape          # e.g., T=100

        # Base names per column, in the same order as kin_data:
        # 1) joint_pairs (L/R averaged)
        # 2) unpaired_cols (as-is)
        col_bases = []
        for col_r, col_l in joint_pairs:
            # Derive a side-agnostic base name from col_r (e.g. "hip_flex_R" -> "hip_flex")
            base = col_r
            if base.lower().endswith("_r") or base.lower().endswith("_l"):
                base = base[:-2]
            col_bases.append(base)

        # Unpaired columns keep their names
        col_bases.extend(unpaired_cols)

        assert len(col_bases) == C, f"len(col_bases)={len(col_bases)} but C={C}"

        # Flattening order: time-major, then columns (matches stride_data.reshape(-1))
        feature_names = [
            f"{base}_t{t:03d}"
            for t in range(T)
            for base in col_bases
        ]

        assert len(feature_names) == X.shape[1], \
            f"feature_names length {len(feature_names)} != X.shape[1] {X.shape[1]}"
        # ---------------------------------------------------------

        data_by_subject[sub] = {
            "X": X,
            "cond_labels": cond_labels,
            "stride_mats": stride_mats,
            "feature_names": feature_names,
        }

        print(
            f"{sub}: "
            f"{X.shape[0]} strides, feature dim = {X.shape[1]} "
        )

    return data_by_subject

def build_kinematics_X_synergy_mode(kinematics_by_subject):
    """
    Kinematic analogue of EMG 'synergy mode':
        Input (per subject, from load_kinematics_subject_data):
            stride_mats: list of (T, C) arrays
            cond_labels: list of stride-level labels (len = n_strides)

        Output (per subject):
            X: (T_total, C)  -- time samples stacked over all strides
            cond_labels: list len = T_total, one label per time sample
    """
    synergy_by_subject = {}

    for sub, d in kinematics_by_subject.items():
        stride_mats = d["stride_mats"]      # list of (T, C)
        stride_cond_labels = d["cond_labels"]

        if not stride_mats:
            print(f"{sub} (kinematics synergy mode): no strides, skipping.")
            continue

        # Infer T, C
        T, C = stride_mats[0].shape

        X_list = []
        cond_labels = []

        for stride_data, cond in zip(stride_mats, stride_cond_labels):
            T_check, C_check = stride_data.shape
            assert (T_check, C_check) == (T, C)
            X_list.append(stride_data)          # (T, C)
            cond_labels.extend([cond] * T)      # one label per time sample

        X = np.vstack(X_list)  # (T_total, C)

        synergy_by_subject[sub] = {
            "X": X,
            "cond_labels": cond_labels,
        }
        

        print(f"{sub} (kinematics synergy mode): X shape = {X.shape}")

    return synergy_by_subject

In [ ]:
# =============================================================================
# GAIT METRICS LOADERS
# =============================================================================
def load_gait_metrics_subject_data():
    """
    Load gait metrics per subject, map to "sXaYbZ",
    and return data_by_subject dict.
    """
    data_by_subject = {}

    for fname in GAIT_METRICS_FILES:
        data_path = GAIT_METRICS_BASE / fname
        if not data_path.exists():
            print(f"Missing gait metrics file: {data_path}, skipping.")
            continue

        df_data = pd.read_excel(data_path)

        sheet_name = fname.split("_")[1].replace(".xlsx", "")

        df_data["Walking Speed"] = df_data["Walking Speed"] \
            .map(WALKING_SPEED_MAPPING).astype(int)
        df_data["Accuracy"] = df_data["Accuracy"] \
            .map(ACCURACY_MAPPING).astype(int)
        df_data["Balance"] = df_data["Balance"] \
            .map(BALANCE_MAPPING).astype(int)

        df_data["Condition"] = df_data.apply(
            lambda row: f"s{row['Walking Speed']:.0f}"
                        f"a{row['Accuracy']:.0f}"
                        f"b{row['Balance']:.0f}",
            axis=1,
        )

        subject_id = sheet_name
        predictors = df_data[GAIT_PREDICTOR_COLUMNS].copy()
        cond_labels = predictors["Condition"].tolist()
        X = predictors.drop(columns=["Condition"]).values

        # store feature names
        feature_names = [c for c in GAIT_PREDICTOR_COLUMNS if c != "Condition"]

        data_by_subject[subject_id] = {
            "X": X,
            "cond_labels": cond_labels,
            "feature_names": feature_names,
        }

        print(
            f"{fname}: "
            f"{X.shape[0]} samples, feature dim = {X.shape[1]} "
        )

    return data_by_subject

In [ ]:
# =============================================================================
# EMG LOADERS
# =============================================================================

def extract_emg_strides(
    subjects,
    drivepath,
    target_conditions,
    trial_names_by_condition,
    base_muscles,
):
    """
    Low-level extraction: returns, per subject, a *list* of stride arrays and labels.

    For each subject:
        strides: list of arrays, each (T, C) where C = len(base_muscles)
        stride_conditions: list of condition strings, length = n_strides

    Does NOT flatten; just returns (time, muscle) matrices per stride.
    """
    data_raw = {}  # subject -> {"strides": list[(T, C)], "stride_conditions": list[str]}

    for sub_idx, sub in enumerate(subjects):
        try:
            emg_parquet_files = list((drivepath / sub).glob("*emg*.parquet"))
            if not emg_parquet_files:
                print(f"[{sub_idx+1}/{len(subjects)}] {sub}: NO EMG PARQUET FILE")
                continue

            df_emg_all = pd.read_parquet(emg_parquet_files[0])
            if "patient" not in df_emg_all.columns:
                raise ValueError(f"'patient' column not found in EMG parquet for {sub}")

            df_emg_all = df_emg_all[df_emg_all["patient"] == sub].copy()
            if df_emg_all.empty:
                print(f"[{sub_idx+1}/{len(subjects)}] {sub}: no rows for this patient, skipping.")
                continue

            all_muscles = base_muscles[:]
            strides = []
            stride_conditions = []

            for cond in target_conditions:
                if sub == "BMH08" and cond == (1, 2, 2) or cond == (0, 2, 2) and sub in ("BMH01", "BMH02", "BMH08"):
                    print(f"[{sub_idx+1}/{len(subjects)}] {sub}: SKIP CONDITION {cond}")
                    continue

                speed_val, acc_val, bal_val = cond
                cond_key = f"s{speed_val}a{int(acc_val)}b{int(bal_val)}"
                trials_for_cond = trial_names_by_condition.get(cond, {})
                trial_id = trials_for_cond.get(sub, None)
                if not trial_id:
                    print(f"[{sub_idx+1}/{len(subjects)}] {sub}: NO TRIAL {cond}")
                    continue

                df_trial = df_emg_all[df_emg_all["trial"] == trial_id].copy()
                if df_trial.empty:
                    print(f"[{sub_idx+1}/{len(subjects)}] {sub}: TRIAL EMPTY {cond}")
                    continue

                for step_id, df_step in df_trial.groupby("step_index_norm"):
                    for side_prefix in ["R", "L"]:
                        df_cyc = df_step[df_step["channel"].str.startswith(side_prefix)]
                        if df_cyc.empty:
                            continue

                        df_cyc = df_cyc.sort_values("gait_percentage")

                        df_wide = df_cyc.pivot_table(
                            index="gait_percentage",
                            columns="channel",
                            values="normalized_filtered_emg",
                        )
                        if df_wide.empty:
                            continue

                        col_map = {col: col[1:] for col in df_wide.columns}  # drop 'R'/'L'
                        df_wide_base = (
                            df_wide.T
                            .groupby(col_map)
                            .mean()
                            .T
                        )

                        if not set(all_muscles).issubset(df_wide_base.columns):
                            continue

                        df_wide_base = df_wide_base[all_muscles]

                        stride_data = df_wide_base.values.astype(float)  # (T, C)
                        stride_data[stride_data < 0.0] = 0.0  # non-negative

                        # add to downsample to 100 points for consistency with kinematics
                        T_target = 100
                        stride_data = resample(stride_data, T_target, axis=0)  # now (100, C)

                        strides.append(stride_data)
                        stride_conditions.append(cond_key)

            if not strides:
                print(f"[{sub_idx+1}/{len(subjects)}] {sub}: NO VALID EMG STRIDES")
                continue

            data_raw[sub] = {
                "strides": strides,
                "stride_conditions": stride_conditions,
            }
            print(f"[{sub_idx+1}/{len(subjects)}] {sub}: {len(strides)} valid EMG strides")

        except Exception as e:
            print(f"[{sub_idx+1}/{len(subjects)}] {sub}: ERROR - {e}")
            continue

    return data_raw


def build_emg_X_synergy_mode(data_raw, base_muscles):
    """
    Construct X for 'synergy' NMF/PCA:
        X: (T * n_strides, C), C = len(base_muscles)
        cond_labels: one label per *row* (time sample), or per stride if you downsample.
    """
    data_by_subject = {}
    C = len(base_muscles)

    for sub, d in data_raw.items():
        strides = d["strides"]             # list of (T, C)
        stride_conditions = d["stride_conditions"]

        # concatenate all strides along time dimension
        X_list = []
        cond_labels = []

        for stride_data, cond in zip(strides, stride_conditions):
            T, C_check = stride_data.shape
            assert C_check == C
            X_list.append(stride_data)          # (T, C)
            cond_labels.extend([cond] * T)      # one label per time sample

        X = np.vstack(X_list)  # (T_total, C)
        data_by_subject[sub] = {
            "X": X,
            "cond_labels": cond_labels,
        }

        print(f"{sub} (synergy mode): X shape = {X.shape}")

    return data_by_subject

def build_emg_X_stride_mode(data_raw, base_muscles):
    """
    Construct X for 'stride' mode:
        X: (n_strides, T * C)
        cond_labels: one label per stride.
        feature_names: length T*C, matching flattening order.
    """
    data_by_subject = {}
    C = len(base_muscles)

    for sub, d in data_raw.items():
        strides = d["strides"]             # list of (T, C)
        stride_conditions = d["stride_conditions"]

        if len(strides) == 0:
            print(f"{sub} (stride mode): no strides, skipping.")
            continue

        # Infer T, C from first stride
        T, C_check = strides[0].shape
        assert C_check == C, f"Channel count mismatch for {sub}: {C_check} != {C}"

        # Build feature_names to match stride_data.reshape(-1) (row-major)
        # Order: t0_m0, t0_m1, ..., t0_m(C-1), t1_m0, ..., t(T-1)_m(C-1)
        feature_names = [
            f"{mus}_t{t:03d}"
            for t in range(T)
            for mus in base_muscles
        ]
        assert len(feature_names) == T * C

        stride_vectors = []
        for stride_data in strides:
            T_check, C_check = stride_data.shape
            assert T_check == T and C_check == C, \
                f"Stride shape mismatch for {sub}: got {(T_check, C_check)}, expected {(T, C)}"
            stride_vectors.append(stride_data.reshape(-1))  # (T*C,)

        X = np.vstack(stride_vectors)  # (n_strides, T*C)
        data_by_subject[sub] = {
            "X": X,
            "cond_labels": stride_conditions,
            "feature_names": feature_names,
        }

        print(f"{sub} (stride mode): X shape = {X.shape}, n_features = {X.shape[1]}")

    return data_by_subject

## Load data for the selected mode

In [ ]:
# =============================================================================
# LOAD SUBJECT DATA FOR THE SELECTED MODE
# =============================================================================

if MODE not in ["kinematics", "gait_metrics", "emg"]:
    raise ValueError(f"Unknown MODE: {MODE}")

data_by_subject = {}
out_file = None

# Build a short config tag for the filename
if USE_FIXED_COMPONENTS:
    comp_tag = f"fixed{N_FIXED_COMPONENTS}"
else:
    # e.g. vaf90_min4 for 0.9 VAF and min 4 comps
    comp_tag = f"vaf{int(VARIANCE_THRESHOLD*100):02d}_min{MIN_COMPONENTS}"


# "kinematics" and "emg" need the stride-level parquet files, which are not
# distributed with this repository. If they are absent, say so and leave
# data_by_subject empty: the cells below then no-op instead of raising.
RAW_DATA_AVAILABLE = RAW_DATA_DIR.is_dir() and any(
    p.is_dir() and p.name.startswith("BMH") for p in RAW_DATA_DIR.iterdir()
)

if MODE in ("kinematics", "emg") and not RAW_DATA_AVAILABLE:
    print(
        f"MODE={MODE!r} needs the stride-level parquet files, which are not\n"
        f"distributed with this repository. Looked for one BMH* subfolder per\n"
        f"subject in: {RAW_DATA_DIR.resolve()}\n"
        f"Point MO_RAW_DATA_DIR at that folder, or set MODE = 'gait_metrics'\n"
        f"to run from the data that is included. Skipping the rest of the notebook."
    )

elif MODE == "kinematics":
    print("Loading kinematics data...")

    # Trial IDs per prompt combination. Only the kinematics and EMG loaders need
    # this, so it is looked up inside those branches rather than up front.
    target_conditions, trial_names_by_condition = build_trial_names_by_condition(
        SUBJECTIVE_XLS,
        KIN_SUBJECTS,
    )

    df_total = load_all_kinematics_strides()

    kin_raw = load_kinematics_subject_data(
        df_total=df_total,
        subjects=KIN_SUBJECTS,
        target_conditions=target_conditions,
        trial_names_by_condition=trial_names_by_condition,
        joint_pairs=JOINT_PAIRS,
        unpaired_cols=unpaired_cols,
    )

    # Decide stride vs synergy mode, analogous to EMG_NMF_MODE
    if COLLAPSE_MODE == "stride":
        # Use the stride-level X directly
        data_by_subject = kin_raw
        # (each sub: X = (n_strides, T*C))
    elif COLLAPSE_MODE == "synergy":
        # Convert to time-sample (synergy) mode
        data_by_subject = build_kinematics_X_synergy_mode(kin_raw)
        comp_tag = f"{comp_tag}_kin_{COLLAPSE_MODE}"
    else:
        raise ValueError(f"Unknown COLLAPSE_MODE: {COLLAPSE_MODE}")

elif MODE == "gait_metrics":
    print("Loading gait metrics data...")
    data_by_subject = load_gait_metrics_subject_data()
    print(f"Prepared gait metrics data for {len(data_by_subject)} subjects.")

elif MODE == "emg":
    print("Loading EMG data from parquet...")

    target_conditions, trial_names_by_condition = build_trial_names_by_condition(
        SUBJECTIVE_XLS,
        KIN_SUBJECTS,
    )

    data_raw_emg = extract_emg_strides(
            subjects=KIN_SUBJECTS,
            drivepath=RAW_DATA_DIR,
            target_conditions=target_conditions,
            trial_names_by_condition=trial_names_by_condition,
            base_muscles=BASE_MUSCLES,
        )
    if COLLAPSE_MODE == "synergy":
        data_by_subject = build_emg_X_synergy_mode(
            data_raw=data_raw_emg,
            base_muscles=BASE_MUSCLES,
        )
        comp_tag = f"{comp_tag}_{COLLAPSE_MODE}"
    elif COLLAPSE_MODE == "stride":
        data_by_subject = build_emg_X_stride_mode(
            data_raw=data_raw_emg,
            base_muscles=BASE_MUSCLES,
        )
    else:
        raise ValueError(f"Unknown COLLAPSE_MODE: {COLLAPSE_MODE}")

out_file = OUTPUT_DIR / f"decomp_{MODE}_{DECOMP_METHOD}_{comp_tag}.csv"

## Run the decomposition

Fits one model per subject, records variance accounted for, averages the
component scores within each condition and writes the result to `out_file`.

In [ ]:
# RUN DECOMPOSITION
# =============================================================================

print("\n================ CONFIG ================")
print(f"MODE                 : {MODE}")
print(f"METHOD               : {DECOMP_METHOD.upper()}")

if USE_FIXED_COMPONENTS:
    print("COMPONENT SELECTION  : fixed number of components")
    print(f"NUMBER OF COMPONENTS : {N_FIXED_COMPONENTS}")
else:
    print("COMPONENT SELECTION : variance threshold")
    print(f"VARIANCE_THRESHOLD  : {VARIANCE_THRESHOLD}")
    print(f"MIN_COMPONENTS      : {MIN_COMPONENTS}")
print("=========================================\n")

all_rows = []
decomp_results = {}

subjects_sorted = sorted(data_by_subject.keys())


for idx, sub in enumerate(subjects_sorted):
    X = data_by_subject[sub]["X"]
    cond_labels = data_by_subject[sub]["cond_labels"]

    n_samples = X.shape[0]
    if n_samples < MIN_SAMPLES_PER_SUBJECT:
        print(f"[{idx+1}/{len(subjects_sorted)}] {sub}: only {n_samples} samples "
              f"(< {MIN_SAMPLES_PER_SUBJECT}) -> SKIP")
        continue

    try:
        model, X_scores, n_keep, var_explained = run_decomposition(
            X, 
            method=DECOMP_METHOD
        )

        decomp_results[sub] = {}

        if DECOMP_METHOD == "pca" and var_explained is not None:
            cum_var_total = var_explained[:n_keep].sum()
            info_str = f"tVAF = {cum_var_total:.2f}"
        else:
            info_str = f"gVAF = {var_explained:.2f}" # for NMF, var_explained is global VAF

        print(
            f"[{idx+1}/{len(subjects_sorted)}] {sub}: "
            f"n_samples = {n_samples}, "
            f"k_kept = {n_keep}, "
            f"{info_str}"
        )

        decomp_results[sub].update({
            "model": model,
            "X": X,
            "X_scores": X_scores,
            "conditions_per_sample": cond_labels,
            "n_samples": n_samples,
            "var_explained": var_explained,
            "n_components_kept": n_keep,
        })

        rows = aggregate_condition_means(
            subject_id=sub,
            X_scores=X_scores,
            cond_labels=cond_labels,
        )
        all_rows.extend(rows)

    except Exception as e:
        print(f"[{idx+1}/{len(subjects_sorted)}] {sub}: ERROR - {e}")
        continue


if not all_rows:
    # Nothing was loaded (e.g. kinematics/EMG without the raw parquet files).
    # Keep df_scores defined so the cells below no-op, and do not overwrite the
    # committed feature CSV with an empty file.
    df_scores = pd.DataFrame()
    print("\nNo component scores produced, so nothing was written.")
else:
    df_scores = pd.DataFrame(all_rows)
    df_scores.to_csv(out_file, index=False)
    print(f"\nWrote subject-condition component scores to:\n{out_file}")


## Variance accounted for by the retained components

In [ ]:
# ================================================================
# RESULTS-CLAIM SUMMARY: "N principal components preserved X% of variance"
# Run this cell right after the RUN DECOMPOSITION cell above, once per
# MODE (kinematics / gait_metrics / emg), with USE_FIXED_COMPONENTS=True
# and N_FIXED_COMPONENTS=5 (or whatever N you are reporting).
#
# It prints a one-line, paper-ready summary (mean +/- SD, and min-max range)
# for the CURRENT modality and appends a row to
# data/results_claim_vaf_summary.csv so that after running
# kinematics, gait_metrics, and emg in turn, that CSV has one row per
# modality -- the full evidence table for the claim.
# ================================================================

if DECOMP_METHOD == "pca" and USE_FIXED_COMPONENTS:
    vaf_pcts = []
    for s, res in decomp_results.items():
        ve = res.get("var_explained")
        nk = res.get("n_components_kept")
        if ve is not None and nk is not None:
            vaf_pcts.append(100 * ve[:nk].sum())
    vaf_pcts = np.array(vaf_pcts)

    if vaf_pcts.size == 0:
        print("No per-subject var_explained found -- did the RUN DECOMPOSITION cell run first?")
    else:
        # Sample SD (ddof=1) across subjects; matches mean +/- SD reporting convention.
        vaf_sd = float(vaf_pcts.std(ddof=1)) if vaf_pcts.size > 1 else 0.0

        summary_row = {
            "Modality": MODE,
            "N_PCs": N_FIXED_COMPONENTS,
            "N_subjects": int(vaf_pcts.size),
            "VAF_min_pct": round(float(vaf_pcts.min()), 1),
            "VAF_max_pct": round(float(vaf_pcts.max()), 1),
            "VAF_mean_pct": round(float(vaf_pcts.mean()), 1),
            "VAF_sd_pct": round(vaf_sd, 1),
        }

        print("\n============ RESULTS-CLAIM SUMMARY ============")
        print(
            f"{MODE.upper()}: {N_FIXED_COMPONENTS} PCs preserved "
            f"{vaf_pcts.mean():.1f} +/- {vaf_sd:.1f}% of input variance "
            f"(range {vaf_pcts.min():.1f}-{vaf_pcts.max():.1f}%, n={vaf_pcts.size} subjects)"
        )
        print("=================================================\n")

        summary_log_path = OUTPUT_DIR / "results_claim_vaf_summary.csv"
        df_line = pd.DataFrame([summary_row])
        file_exists = summary_log_path.exists()

        # Replace any existing row for this Modality+N_PCs combo, then append fresh
        if file_exists:
            df_prev = pd.read_csv(summary_log_path)
            df_prev = df_prev[
                ~((df_prev["Modality"] == MODE) & (df_prev["N_PCs"] == N_FIXED_COMPONENTS))
            ]
            df_out = pd.concat([df_prev, df_line], ignore_index=True)
        else:
            df_out = df_line

        df_out.to_csv(summary_log_path, index=False)
        print(f"Updated: {summary_log_path}")
        print(df_out.to_string(index=False))
else:
    print("Skipping results-claim summary: requires DECOMP_METHOD='pca' and USE_FIXED_COMPONENTS=True.")